In [20]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API+KEY"]=os.getenv("OPENAI_API_KEY")




## To load the file

In [9]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader("heart_disease_reference.pdf")
docs=loader.load()
print(docs)

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-28T21:33:31+00:00', 'author': 'Educational Reference Document', 'keywords': '', 'moddate': '2026-04-28T21:33:31+00:00', 'subject': '(unspecified)', 'title': 'Heart Disease: A Comprehensive Reference', 'trapped': '/False', 'source': 'heart_disease_reference.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content="Heart Disease: A Comprehensive Reference\nAn educational reference covering coronary artery disease, heart failure, arrhythmias, valvular disease, diagnosis,\ntreatment, and prevention. Compiled for educational and informational use only. This document is not medical advice.\n1. Overview of Cardiovascular Disease\nCardiovascular disease (CVD) is an umbrella term for disorders of the heart and blood vessels. It is the\nleading cause of death worldwide, responsible for an estimated 17.9 million deaths each year according to\nthe World Health Org

## Splitting the document into chunks


In [10]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
textsplit=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
text_split=textsplit.split_documents(docs)
print(text_split)

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-04-28T21:33:31+00:00', 'author': 'Educational Reference Document', 'keywords': '', 'moddate': '2026-04-28T21:33:31+00:00', 'subject': '(unspecified)', 'title': 'Heart Disease: A Comprehensive Reference', 'trapped': '/False', 'source': 'heart_disease_reference.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content='Heart Disease: A Comprehensive Reference\nAn educational reference covering coronary artery disease, heart failure, arrhythmias, valvular disease, diagnosis,\ntreatment, and prevention. Compiled for educational and informational use only. This document is not medical advice.\n1. Overview of Cardiovascular Disease\nCardiovascular disease (CVD) is an umbrella term for disorders of the heart and blood vessels. It is the\nleading cause of death worldwide, responsible for an estimated 17.9 million deaths each year according to\nthe World Health Org

## Embeddings 

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},
)
vectorstore=Chroma.from_documents(documents=text_split,embedding=embeddings,persist_directory="./chroma_db")
retriever=vectorstore.as_retriever(search_kwargs={"k":8})
print(embeddings)
print(vectorstore)
print(retriever)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5758.43it/s]


model_name='BAAI/bge-base-en-v1.5' cache_folder=None model_kwargs={} encode_kwargs={'normalize_embeddings': True} query_encode_kwargs={} multi_process=False show_progress=False
tags=['Chroma', 'HuggingFaceEmbeddings'] vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000023795BA11E0> search_kwargs={'k': 8}


In [12]:
!pip install rank_bm25

## Hybrid Retreiver

In [13]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers import BM25Retriever

bm25_retriever=BM25Retriever.from_documents(text_split)
bm25_retriever.k=8

retriever=vectorstore.as_retriever(search_kwargs={"k":4}) ## existing vectorstore

hybrid_retriever=EnsembleRetriever(retrievers=[bm25_retriever,retriever],weights=[0.4,0.6])


## ReRanking(Cross_encoder)

In [14]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant",temperature=0.1)

cross_encoder=HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")

reranker=CrossEncoderReranker(model=cross_encoder,top_n=3)

final_retreiver=ContextualCompressionRetriever(base_compressor=reranker,base_retriever=hybrid_retriever)
multi_query_retreiver=MultiQueryRetriever.from_llm(retriever=final_retreiver,llm=model)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 14382.68it/s]


In [15]:

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough


parser=StrOutputParser()


parser = StrOutputParser()

model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.1,
)

Prompt = ChatPromptTemplate.from_messages([
    ("system", """you are a helpful medical assistant.Answer ONLY using the provided context.Do NOT use external knowledge.If the answer is not in the context,say:"Not Available".
    Always rely strictly onm context
     Context:
{context}"""),
    ("user", """Answer the following question ONLY from the context.question: {question}""")
])

rag_chain = (
    {
        "context": multi_query_retreiver,
        "question": RunnablePassthrough()
    }
    | Prompt
    | model
    | parser
)

print(rag_chain.invoke("What is the NYHA classification for heart failure?"))

The New York Heart Association (NYHA) classification grades symptom severity. Class I patients have no limitation of physical activity. Class II have slight limitation; ordinary activity causes symptoms. Class III patients have marked limitation; less than ordinary activity causes symptoms.


In [16]:
questions = [
    "What is the NYHA classification for heart failure?",
    "What causes shortness of breath when lying down?",
    "What is the CHA2DS2-VASc score threshold for anticoagulation?",
    "What's the best treatment for diabetes?", ]

for i in questions:
    print(f"\n{'='*70}\nQ: {i}\n{'='*70}")
    answer = rag_chain.invoke(i)
    print(answer)


Q: What is the NYHA classification for heart failure?
The New York Heart Association (NYHA) classification grades symptom severity. Class I patients have no limitation of physical activity. Class II have slight limitation; ordinary activity causes symptoms. Class III patients have marked limitation; less than ordinary activity causes symptoms. Class IV patients have severe limitation; no ordinary activity causes symptoms.

Q: What causes shortness of breath when lying down?
Not Available

Q: What is the CHA2DS2-VASc score threshold for anticoagulation?
A score of 2 or more in men or 3 or more in women generally warrants oral anticoagulation.

Q: What's the best treatment for diabetes?
Not Available


## Model Evaluation using RAGAS 

In [17]:
!pip install ragas

In [18]:
questions = [
    "What is the NYHA classification for heart failure?",
    "What causes shortness of breath when lying down?",
    "What is the CHA2DS2-VASc score threshold for anticoagulation?",
    "What's the best treatment for diabetes?", ]
ground_truths = [
    "NYHA Class I has no limitation of physical activity. Class II has slight limitation. Class III has marked limitation. Class IV has symptoms at rest and inability to perform physical activity without discomfort.",
    "Shortness of breath when lying down is called orthopnea and is listed as a common symptom of heart failure.",
    "A CHA2DS2-VASc score of 2 or more in men or 3 or more in women generally warrants oral anticoagulation.",
    "Not available in the provided context. The document discusses cardiovascular risk reduction in diabetes, not the best overall diabetes treatment."]

answers=[]
contexts=[]

for i in questions:
    docs=multi_query_retreiver.invoke(i)
    contexts.append([doc.page_content for doc in docs])

    answer=rag_chain.invoke(i)
    answers.append(answer)

In [21]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

dataset = Dataset.from_dict({
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths
})

result = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],embeddings=embeddings
)
print(result)

C:\Users\moham\AppData\Local\Temp\ipykernel_21392\284307369.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\moham\AppData\Local\Temp\ipykernel_21392\284307369.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\moham\AppData\Local\Temp\ipykernel_21392\284307369.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ra

{'faithfulness': 0.6667, 'answer_relevancy': 0.3906, 'context_precision': 0.3194, 'context_recall': 0.7500}
